In [12]:
# клонируем репозиторий yolov7
! git clone https://github.com/WongKinYiu/yolov7.git

Cloning into 'yolov7'...


In [ ]:
# устанавливаем необходимые библиотеки
%cd yolov7
!pip install -r requirements.txt

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  

yolov7_pose_url = os.getenv('YOLOV7_POSE_URL')
test_img1 = os.getenv('TEST_IMG1_URL')
test_img2 = os.getenv('TEST_IMG2_URL')

In [14]:
import requests
WEIGHTS_URL = 'https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-w6-pose.pt'
open(yolov7_pose_url, "wb").write(requests.get(WEIGHTS_URL).content)

161114789

In [4]:
import cv2
import torch
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

from models.experimental import attempt_load 
from utils.datasets import letterbox 
from utils.general import non_max_suppression_kpt, xywh2xyxy
from utils.plots import output_to_keypoint, plot_skeleton_kpts, plot_one_box

In [ ]:
from models.yolo import Model
from models.common import Conv

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('Device:', device)
torch.serialization.add_safe_globals([Model, Conv])

model = attempt_load(yolov7_pose_url, map_location=device) 
model.eval()

print('Number of classes:', model.yaml['nc'])
print('Number of keypoints:', model.yaml['nkpt'])

Device: cpu
Fusing layers... 
Number of classes: 1
Number of keypoints: 17


In [ ]:
def show_image(img, figsize=(6,6)):            
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# read original image
orig_img = cv2.imread(test_img1)

orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
print('Original image', orig_img.shape)
show_image(orig_img)

Original image (1280, 960, 3)


C:\Users\Alex\AppData\Local\Temp\ipykernel_11800\2249939956.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
img = letterbox(orig_img, 640, stride=64, auto=True)[0]
print('Resized image', img.shape)
show_image(img)

Resized image (640, 512, 3)


C:\Users\Alex\AppData\Local\Temp\ipykernel_11800\2249939956.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
print('Anchors:', model.yaml['anchors'])

Anchors: [[19, 27, 44, 40, 38, 94], [96, 68, 86, 152, 180, 137], [140, 301, 303, 264, 238, 542], [436, 615, 739, 380, 925, 792]]


In [38]:
img_ = transforms.ToTensor()(img)
# add dimension
img_ = torch.unsqueeze(img_, 0)
print('Transformed to tensor image:', img_.shape)
# send the picture to the calculating device
img_ = img_.to(device).float()

with torch.no_grad():
    pred, _ = model(img_)
print('Predictions shape:', pred.shape)

Transformed to tensor image: torch.Size([1, 3, 640, 512])
Predictions shape: torch.Size([1, 20400, 57])


In [39]:
pred = non_max_suppression_kpt(pred, 
                               conf_thres=0.25, 
                               iou_thres=0.65, 
                               nc=model.yaml['nc'], 
                               nkpt=model.yaml['nkpt'], 
                               kpt_label=True)
print('Detected poses:', len(pred))
print('Prediction shape:', pred[0].shape)

Detected poses: 1
Prediction shape: torch.Size([1, 57])


In [32]:
!pip install PyQt5

   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/6.9 MB 8.5 MB/s eta 0:00:01
   ---------- ----------------------------- 1.8/6.9 MB 9.1 MB/s eta 0:00:01
   --------------------------- ------------ 4.7/6.9 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------  6.8/6.9 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 9.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/50.1 MB ? eta -:--:--
   - -------------------------------------- 1.3/50.1 MB 9.6 MB/s eta 0:00:06
   --- ------------------------------------ 4.5/50.1 MB 10.3 MB/s eta 0:00:05
   ----- ---------------------------------- 6.8/50.1 MB 10.8 MB/s eta 0:00:05
   ------- -------------------------------- 9.2/50.1 MB 11.0 MB/s eta 0:00:04
   --------- ------------------------------ 11.3/50.1 MB 10.8 MB/s eta 0:00:04
   ---------- ----------------------------- 13.6/50.1 MB 10.8 MB/s eta 0:00:04
   ---------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import matplotlib
matplotlib.use('Qt5Agg')

In [ ]:
def plot_pose_prediction(img : cv2.Mat, pred : list, thickness=2, 
                         show_bbox : bool=True) -> cv2.Mat:
    bbox = xywh2xyxy(pred[:,2:6])
    for idx in range(pred.shape[0]):
        plot_skeleton_kpts(img, pred[idx, 7:].T, 3)
        if show_bbox:
            plot_one_box(bbox[idx], img, line_thickness=thickness)

pred = output_to_keypoint(pred)
plot_pose_prediction(img, pred)
show_image(img)
plt.show()